<a href="https://colab.research.google.com/github/Nanda-Lopes/AlgoStudies/blob/main/Stanford_Algorithms_Specialization_Course4_W1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução e Algoritmo de Johnson

# Problema do Caminho Mínimo entre Todos os Pares (APSP)

O objetivo desta tarefa é calcular o menor caminho mínimo entre todos os pares de vértices ($\min_{u,v \in V} d(u,v)$) para três grafos direcionados com pesos que podem ser negativos (`g1.txt`, `g2.txt`, `g3.txt`).

Caso um grafo contenha ciclos de custo negativo, o cálculo de caminhos mínimos torna-se indefinido para esse grafo. Se todos os três grafos possuírem ciclos negativos, a resposta é `NULL`. Caso contrário, a resposta corresponde ao menor valor entre as distâncias mínimas de todos os grafos válidos.

Para obter eficiência assintótica máxima em grafos esparsos ($O(V \cdot E \log V)$), o **Algoritmo de Johnson** foi implementado:
1. Um vértice fantasma $s$ foi adicionado com arestas direcionadas de peso zero para todos os vértices do grafo original.
2. O algoritmo de Bellman-Ford foi executado a partir de $s$ para computar os potenciais $p(v)$ de cada vértice. Se um ciclo negativo foi detectado, o grafo foi descartado.
3. As arestas originais foram reponderadas: $w'(u,v) = w(u,v) + p(u) - p(v) \ge 0$.
4. O algoritmo de Dijkstra com fila de prioridade (Min-Heap) foi executado a partir de cada vértice $u$ no grafo reponderado.
5. As distâncias reais foram recuperadas via $d(u,v) = d'(u,v) - p(u) + p(v)$, e o menor valor global foi armazenado.

In [1]:
import heapq

def bellman_ford(num_vertices, edges, source):
    dist = [float('inf')] * (num_vertices + 1)
    dist[source] = 0
    for _ in range(num_vertices):
        updated = False
        for u, v, w in edges:
            if dist[u] != float('inf') and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                updated = True
        if not updated:
            break
    else:
        for u, v, w in edges:
            if dist[u] != float('inf') and dist[u] + w < dist[v]:
                return None
    return dist

def dijkstra(num_vertices, adj, source):
    dist = [float('inf')] * (num_vertices + 1)
    dist[source] = 0
    heap = [(0, source)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(heap, (dist[v], v))
    return dist

def solve_apsp(filename):
    with open(filename, "r") as f:
        header = f.readline()
        num_vertices, num_edges = map(int, header.split())
        edges = []
        for line in f:
            if line.strip():
                u, v, w = map(int, line.split())
                edges.append((u, v, w))
    extended_edges = list(edges)
    for v in range(1, num_vertices + 1):
        extended_edges.append((0, v, 0))
    potentials = bellman_ford(num_vertices, extended_edges, 0)
    if potentials is None:
        return None
    adj = [[] for _ in range(num_vertices + 1)]
    for u, v, w in edges:
        w_prime = w + potentials[u] - potentials[v]
        adj[u].append((v, w_prime))
    min_shortest_path = float('inf')
    for u in range(1, num_vertices + 1):
        d_prime = dijkstra(num_vertices, adj, u)
        for v in range(1, num_vertices + 1):
            if u != v and d_prime[v] != float('inf'):
                real_dist = d_prime[v] - potentials[u] + potentials[v]
                if real_dist < min_shortest_path:
                    min_shortest_path = real_dist
    return min_shortest_path

# Processamento dos Grafos e Determinação do Valor Ótimo

Nesta etapa, o Algoritmo de Johnson foi executado sobre os arquivos `g1.txt`, `g2.txt` e `g3.txt`.

Os resultados intermediários identificam quais grafos possuem ciclos negativos e determinam o menor caminho mínimo entre todos os grafos válidos.

In [2]:
files = ["g1.txt", "g2.txt", "g3.txt"]
results = {}

for filename in files:
    res = solve_apsp(filename)
    results[filename] = res
    if res is None:
        print(f"{filename}: Ciclo de custo negativo detectado (NULL)")
    else:
        print(f"{filename}: Menor caminho mínimo = {res}")

valid_results = [v for v in results.values() if v is not None]

print("==================================")
print("RESPOSTA FINAL (SHORTEST SHORTEST PATH):")
if not valid_results:
    print("NULL")
else:
    print(min(valid_results))
print("==================================")

g1.txt: Ciclo de custo negativo detectado (NULL)
g2.txt: Ciclo de custo negativo detectado (NULL)
g3.txt: Menor caminho mínimo = -19
RESPOSTA FINAL (SHORTEST SHORTEST PATH):
-19
